# Exploratory Data Analysis - Canadian Real Estate

This notebook explores the merged dataset to understand:
- Price trends by city and property type
- Correlations between features
- Missing data patterns
- Regime changes (COVID, rate hikes)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Load data
data_dir = Path('../data/processed')
merged_path = data_dir / 'merged_data.csv'

if merged_path.exists():
    df = pd.read_csv(merged_path, parse_dates=['date'])
    print(f"Loaded {len(df)} rows")
else:
    print("Run: python src/ingest.py first")

## Dataset Overview

In [ ]:
# Basic info
print(f"Shape: {df.shape}")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"\nCities: {df['city'].unique().tolist()}")
print(f"\nProperty types: {df['property_type'].unique().tolist()}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# Missing data
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

missing_df[missing_df['Missing Count'] > 0]

## Price Trends by City

In [ ]:
# Average price by city over time (all property types)
city_prices = df.groupby(['date', 'city'])['benchmark_price'].mean().reset_index()

fig = px.line(
    city_prices,
    x='date',
    y='benchmark_price',
    color='city',
    title='Average Benchmark Price by City (All Property Types)',
    labels={'benchmark_price': 'Price ($)', 'date': 'Date'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Price by city and property type
fig = px.line(
    df,
    x='date',
    y='benchmark_price',
    color='property_type',
    facet_col='city',
    title='Benchmark Price by City and Property Type',
    labels={'benchmark_price': 'Price ($)', 'date': 'Date'}
)
fig.update_layout(height=600)
fig.show()

## Price Changes Over Time

In [ ]:
# Calculate YoY change
df['price_yoy'] = df.groupby(['city', 'property_type'])['benchmark_price'].pct_change(12)

# Plot YoY change
yoy_df = df.dropna(subset=['price_yoy'])

fig = px.line(
    yoy_df,
    x='date',
    y='price_yoy',
    color='city',
    facet_col='property_type',
    title='Year-over-Year Price Change by City',
    labels={'price_yoy': 'YoY Change (%)', 'date': 'Date'}
)
fig.update_layout(height=500)
fig.update_traces(line=dict(width=2))
fig.show()

## Interest Rates and Price Relationship

In [ ]:
# Plot rates over time
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['date'].unique(),
    y=df.groupby('date')['overnight_rate'].first(),
    name='Overnight Rate',
    line=dict(color='red', width=2)
))

fig.add_trace(go.Scatter(
    x=df['date'].unique(),
    y=df.groupby('date')['prime_rate'].first(),
    name='Prime Rate',
    line=dict(color='blue', width=2)
))

fig.update_layout(
    title='Bank of Canada Interest Rates Over Time',
    xaxis_title='Date',
    yaxis_title='Rate (%)',
    height=400
)
fig.show()

In [ ]:
# Scatter plot: rates vs prices
scatter_df = df.dropna(subset=['overnight_rate', 'benchmark_price'])

fig = px.scatter(
    scatter_df,
    x='overnight_rate',
    y='benchmark_price',
    color='city',
    size='sales',
    hover_data=['date', 'property_type'],
    title='Overnight Rate vs Benchmark Price',
    labels={'overnight_rate': 'Overnight Rate (%)', 'benchmark_price': 'Price ($)'}
)
fig.update_layout(height=500)
fig.show()

## Supply & Demand Metrics

In [ ]:
# Sales to active ratio over time
fig = px.line(
    df,
    x='date',
    y='sales_to_active_ratio',
    color='city',
    facet_col='property_type',
    title='Sales to Active Ratio (Market Tightness)',
    labels={'sales_to_active_ratio': 'Ratio', 'date': 'Date'}
)
fig.add_hline(y=0.12, line_dash='dash', annotation_text="Buyer's Market")
fig.add_hline(y=0.20, line_dash='dash', annotation_text="Seller's Market")
fig.update_layout(height=500)
fig.show()

In [ ]:
# Days on market
fig = px.box(
    df,
    x='city',
    y='days_on_market',
    color='property_type',
    title='Days on Market Distribution by City and Property Type',
    labels={'days_on_market': 'Days', 'city': 'City'}
)
fig.update_layout(height=400)
fig.show()

## Rental Market

In [ ]:
# Average rent by city
rent_df = df.groupby(['date', 'city'])['avg_rent_2br'].mean().reset_index()

fig = px.line(
    rent_df,
    x='date',
    y='avg_rent_2br',
    color='city',
    title='Average 2BR Rent by City',
    labels={'avg_rent_2br': 'Monthly Rent ($)', 'date': 'Date'}
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Price to rent ratio
df['price_to_rent_ratio'] = df['benchmark_price'] / (df['avg_rent_2br'] * 12)

fig = px.box(
    df.dropna(subset=['price_to_rent_ratio']),
    x='city',
    y='price_to_rent_ratio',
    color='property_type',
    title='Price to Rent Ratio by City and Property Type',
    labels={'price_to_rent_ratio': 'Ratio', 'city': 'City'}
)
fig.update_layout(height=400)
fig.show()

## Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove columns with too much missing data
numeric_cols = [c for c in numeric_cols if df[c].isna().sum() / len(df) < 0.5]

corr_df = df[numeric_cols].dropna()

# Calculate correlation matrix
corr_matrix = corr_df.corr()

# Plot heatmap
plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1
)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with benchmark price
price_corr = corr_matrix['benchmark_price'].sort_values(ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(
    x=price_corr.values[1:15],  # Exclude self-correlation
    y=price_corr.index[1:15]
)
plt.title('Top 15 Features Correlated with Benchmark Price')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

## Seasonality Analysis

In [ ]:
# Add month column
df['month'] = df['date'].dt.month

# Price by month
monthly_price = df.groupby('month')['benchmark_price'].agg(['mean', 'std']).reset_index()
monthly_price['cv'] = monthly_price['std'] / monthly_price['mean']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly_price['month'],
    y=monthly_price['mean'],
    name='Avg Price',
    marker_color='steelblue'
))
fig.update_layout(
    title='Average Price by Month (Seasonality)',
    xaxis_title='Month',
    yaxis_title='Price ($)',
    height=400
)
fig.show()

In [ ]:
# Sales by month
monthly_sales = df.groupby('month')['sales'].mean().reset_index()

fig = px.bar(
    monthly_sales,
    x='month',
    y='sales',
    title='Average Sales by Month',
    labels={'month': 'Month', 'sales': 'Avg Sales'}
)
fig.update_layout(height=400)
fig.show()

## Regime Change Analysis

In [ ]:
# Define regimes
def get_regime(date):
    if date < pd.Timestamp('2020-03-01'):
        'Pre-COVID'
    elif date < pd.Timestamp('2022-01-01'):
        return 'COVID Era'
    elif date < pd.Timestamp('2022-07-01'):
        return 'Hot Market'
    elif date < pd.Timestamp('2023-07-01'):
        return 'Rate Hikes'
    else:
        return 'Post-Hike'

df['regime'] = df['date'].apply(get_regime)

# Price stats by regime
regime_stats = df.groupby('regime')['benchmark_price'].agg(['mean', 'median', 'std', 'count']).reset_index()
regime_stats

In [ ]:
# Plot price distribution by regime
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df.dropna(subset=['benchmark_price']),
    x='regime',
    y='benchmark_price',
    hue='property_type'
)
plt.title('Price Distribution by Market Regime')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
# Overall summary
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"\nTotal rows: {len(df):,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nCities: {', '.join(df['city'].unique())}")
print(f"Property types: {', '.join(df['property_type'].unique())}")
print(f"\nBenchmark Price Statistics:")
print(f"  Mean: ${df['benchmark_price'].mean():,.0f}")
print(f"  Median: ${df['benchmark_price'].median():,.0f}")
print(f"  Std: ${df['benchmark_price'].std():,.0f}")
print(f"  Min: ${df['benchmark_price'].min():,.0f}")
print(f"  Max: ${df['benchmark_price'].max():,.0f}")

In [ ]:
# By city summary
city_summary = df.groupby('city')['benchmark_price'].agg(['mean', 'median', 'std', 'count']).round(0)
city_summary.columns = ['Mean Price', 'Median Price', 'Std Dev', 'Count']
city_summary = city_summary.sort_values('Mean Price', ascending=False)
city_summary